# Raw media 업로드

`tmdb_movie_data.csv`, `aladin_webtoon_data.csv` → `POST /admin/media/movies`, `POST /admin/media/webtoons`

웹툰 `id`는 알라딘 링크의 `ItemId` 쿼리 파라미터에서 추출합니다.

In [ ]:
import html
import os
import re
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "raw_data"
REPO_ROOT = PROJECT_DIR.parent
ENV_FILE = REPO_ROOT / ".env"

MOVIE_CSV = DATA_DIR / "tmdb_movie_data.csv"
WEBTOON_CSV = DATA_DIR / "aladin_webtoon_data.csv"

UPLOAD_CHUNK = 500
REQUEST_TIMEOUT = 30  # 초 — 길면 노트북 인터럽트가 안 먹힘

In [6]:
ITEM_ID_RE = re.compile(r"ItemId=(\d+)", re.IGNORECASE)


def txt(value) -> str:
    return "" if pd.isna(value) else str(value).strip()


def opt_str(value) -> str | None:
    s = txt(value)
    return s or None


def parse_webtoon_id(link: str) -> int:
    m = ITEM_ID_RE.search(txt(link))
    if not m:
        raise ValueError(f"ItemId 없음: {link!r}")
    return int(m.group(1))


def movie_row_to_item(row) -> dict:
    return {
        "id": int(row["movie_id"]),
        "title": txt(row["title"]),
        "genres": txt(row["genres"]),
        "overview": txt(row["overview"]),
        "keywords": opt_str(row["keywords"]),
        "director": txt(row["director"]),
        "cast": opt_str(row["cast"]),
        "runtime": int(row["runtime"]),
        "popularity": float(row["popularity"]),
        "vote_average": float(row["vote_average"]),
        "vote_count": int(row["vote_count"]),
        "release_date": txt(row["release_date"]),
        "poster_url": txt(row["poster_url"]),
    }


def webtoon_row_to_item(row) -> dict:
    desc = opt_str(row["description"])
    if desc:
        desc = html.unescape(desc)
    return {
        "id": parse_webtoon_id(row["link"]),
        "series_title": txt(row["series_title"]),
        "genres": txt(row["genres"]),
        "author": txt(row["author"]),
        "sales_point": int(row["sales_point"]),
        "description": desc,
        "publisher": txt(row["publisher"]),
        "pub_date": txt(row["pub_date"]),
        "cover_url": txt(row["cover_url"]),
        "link": txt(row["link"]),
        "original_title": txt(row["original_title"]),
    }

In [7]:
movies_df = pd.read_csv(MOVIE_CSV)
webtoons_df = pd.read_csv(WEBTOON_CSV)

movie_items = [movie_row_to_item(r) for _, r in movies_df.iterrows()]
webtoon_items = [webtoon_row_to_item(r) for _, r in webtoons_df.iterrows()]

print(f"영화 {len(movie_items)}건, 웹툰 {len(webtoon_items)}건")
print("영화 샘플:", movie_items[0])
print("웹툰 샘플:", webtoon_items[0])

영화 1609건, 웹툰 309건
영화 샘플: {'id': 1304313, 'title': '리 크로닌의 미이라', 'genres': '스릴러/범죄, 공포/호러', 'overview': '한 기자의 어린 딸이 흔적도 없이 사막으로 사라진다. 그리고 8년 후, 산산이 부서진 가족은 그녀가 다시 돌아왔다는 소식에 충격을 받는다. 하지만 기쁨으로 가득해야 할 재회는 곧 살아 있는 악몽으로 변해간다.', 'keywords': 'journalist, egypt, monster, ritual, kidnapping', 'director': '리 크로닌', 'cast': '잭 레이너, 라이아 코스타, 메이 칼라마위', 'runtime': 133, 'popularity': 604.065, 'vote_average': 8.0, 'vote_count': 795, 'release_date': '2026-04-15', 'poster_url': 'https://image.tmdb.org/t/p/w500/zXlwtzXP80yc3X0VokdxvmsFQfw.jpg'}
웹툰 샘플: {'id': 393727271, 'series_title': '전지적 독자 시점', 'genres': '판타지', 'author': '슬리피-C (지은이), 싱숑 (원작), UMI (각색)', 'sales_point': 18910, 'description': '낙원의 멸망을 딛고 비상한 키메라 드래곤. 그리고 일행 앞에 주어진 새로운 시련. 김독자는 가장 사랑하는 존재에게 죽게 될 것이다. 예언대로 됐을 뿐이야. 독자야, 나는 세상 누구보다 너를 사랑한단다. 어쩌면 나 자신보다 더. 지금부터 모든 걸 ‘다시 읽을’ 거란다.', 'publisher': '에이템포미디어', 'pub_date': '2026-05-22', 'cover_url': 'https://image.aladin.co.kr/product/39372/72/coversum/k782139662_1.jpg', 'link': 'https://www.aladin.

## API 업로드

리포지토리 루트 `.env`에 `BASE_URL`, `ADMIN_API_KEY` 필요.

In [8]:
load_dotenv(ENV_FILE)
BASE_URL = os.environ["BASE_URL"].rstrip("/")
ADMIN_API_KEY = os.environ["ADMIN_API_KEY"]
API_HEADERS = {"X-API-Key": ADMIN_API_KEY}


def post_bulk(path: str, items: list[dict]) -> int:
    if not items:
        return 0
    url = f"{BASE_URL}/admin/media{path}"
    total = 0
    for i in range(0, len(items), UPLOAD_CHUNK):
        chunk = items[i : i + UPLOAD_CHUNK]
        resp = requests.post(
            url,
            json={"items": chunk},
            headers=API_HEADERS,
            timeout=REQUEST_TIMEOUT,
        )
        if not resp.ok:
            raise RuntimeError(f"{resp.status_code} {path}: {resp.text}")
        total += resp.json()["upserted"]
        print(f"{path} chunk {i // UPLOAD_CHUNK + 1}: +{resp.json()['upserted']} (누적 {total})")
    return total


n_movies = post_bulk("/movies", movie_items)
n_webtoons = post_bulk("/webtoons", webtoon_items)
print(f"\n완료: movies={n_movies}, webtoons={n_webtoons}")

/movies chunk 1: +500 (누적 500)
/movies chunk 2: +500 (누적 1000)
/movies chunk 3: +500 (누적 1500)
/movies chunk 4: +109 (누적 1609)
/webtoons chunk 1: +309 (누적 309)

완료: movies=1609, webtoons=309
